# KITTI 3D Object Detection — Dataset İndirme

Bu notebook KITTI datasetini Colab üzerinden indirir ve Google Drive'a kaydeder.

**Bir kere çalıştırman yeterli.** İndirdikten sonra Drive'da kalır, tekrar indirmene gerek yok.

### İndirilecekler (~13GB):
- Velodyne point clouds (LiDAR verisi) — 29GB sıkıştırılmış
- Calibration dosyaları — küçük
- Label dosyaları (3D bounding box GT) — küçük
- Left color images — 12GB sıkıştırılmış (opsiyonel, camera fusion için)

### Önemli:
KITTI indirmek için hesap açman lazım (ücretsiz):
https://www.cvlibs.net/datasets/kitti/user_register.php

In [ ]:
# GPU'ya gerek yok, bu notebook sadece indirme yapıyor
# Runtime > Change runtime type > None (GPU kullanma, kredi harcama)

from google.colab import drive
drive.mount('/content/drive')

import os

# Drive'daki hedef klasör
PROJECT_DIR = '/content/drive/MyDrive/uncertainty_3d_detection'
KITTI_DIR = f'{PROJECT_DIR}/data/kitti'
TRAIN_DIR = f'{KITTI_DIR}/training'

os.makedirs(f'{TRAIN_DIR}/velodyne', exist_ok=True)
os.makedirs(f'{TRAIN_DIR}/calib', exist_ok=True)
os.makedirs(f'{TRAIN_DIR}/label_2', exist_ok=True)
os.makedirs(f'{TRAIN_DIR}/image_2', exist_ok=True)
os.makedirs(f'{KITTI_DIR}/ImageSets', exist_ok=True)

print(f'Hedef klasör: {KITTI_DIR}')
print('Klasörler oluşturuldu.')

## Yöntem 1: KITTI Sitesinden Manuel İndirme (Önerilen)

1. https://www.cvlibs.net/datasets/kitti/eval_object.php?obj_benchmark=3d adresine git
2. Hesap oluştur / giriş yap
3. Aşağıdaki dosyaları indir:
   - **data_object_velodyne.zip** (Velodyne point clouds, 29GB) — ZORUNLU
   - **data_object_calib.zip** (Calibration) — ZORUNLU
   - **data_object_label_2.zip** (Labels) — ZORUNLU
   - **data_object_image_2.zip** (Left color images, 12GB) — İSTEĞE BAĞLI
4. İndirilen zip dosyalarını Google Drive'a yükle (herhangi bir yere)
5. Aşağıdaki hücrede zip path'lerini güncelle ve çalıştır

In [ ]:
# ============================================================
# ZIP DOSYALARININ DRIVE'DAKİ PATH'LERİNİ BURAYA YAZ
# İndirip Drive'a attıktan sonra path'leri güncelle
# ============================================================

VELODYNE_ZIP = '/content/drive/MyDrive/data_object_velodyne.zip'    # ZORUNLU
CALIB_ZIP    = '/content/drive/MyDrive/data_object_calib.zip'       # ZORUNLU
LABEL_ZIP    = '/content/drive/MyDrive/data_object_label_2.zip'     # ZORUNLU
IMAGE_ZIP    = '/content/drive/MyDrive/data_object_image_2.zip'     # OPSİYONEL

# Kontrol
for name, path in [('Velodyne', VELODYNE_ZIP), ('Calib', CALIB_ZIP), ('Label', LABEL_ZIP)]:
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / 1e9
        print(f'  {name}: BULUNDU ({size_gb:.1f} GB)')
    else:
        print(f'  {name}: BULUNAMADI — {path}')
        print(f'         Lütfen KITTI sitesinden indirip Drive\'a yükle.')

if os.path.exists(IMAGE_ZIP):
    size_gb = os.path.getsize(IMAGE_ZIP) / 1e9
    print(f'  Image:  BULUNDU ({size_gb:.1f} GB) — opsiyonel, güzel')
else:
    print(f'  Image:  Yok (opsiyonel, şimdilik sorun değil)')

In [ ]:
%%time
# ============================================================
# ZIP AÇMA — Colab local'e aç, sonra Drive'a taşı
# Local disk daha hızlı, Drive'a direkt unzip çok yavaş
# ============================================================

TEMP_DIR = '/content/kitti_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

# --- 1. Calibration (küçük, hızlı) ---
if not os.path.exists(f'{TRAIN_DIR}/calib/000000.txt'):
    print('Calibration açılıyor...')
    !unzip -o -q {CALIB_ZIP} -d {TEMP_DIR}
    !mv {TEMP_DIR}/training/calib/* {TRAIN_DIR}/calib/
    print('  Calib tamam.')
else:
    print('Calib zaten mevcut, atlanıyor.')

# --- 2. Labels (küçük, hızlı) ---
if not os.path.exists(f'{TRAIN_DIR}/label_2/000000.txt'):
    print('Labels açılıyor...')
    !unzip -o -q {LABEL_ZIP} -d {TEMP_DIR}
    !mv {TEMP_DIR}/training/label_2/* {TRAIN_DIR}/label_2/
    print('  Labels tamam.')
else:
    print('Labels zaten mevcut, atlanıyor.')

print('\nKüçük dosyalar tamamlandı.')

In [ ]:
%%time
# --- 3. Velodyne point clouds (BÜYÜK, ~29GB zip → ~12GB açılmış) ---
# Bu adım 15-30 dakika sürebilir

velodyne_check = f'{TRAIN_DIR}/velodyne/000000.bin'

if not os.path.exists(velodyne_check):
    print('Velodyne açılıyor... (bu uzun sürer, ~15-30 dk)')
    print('Colab session kapanmasın diye bu sekmeyi açık tut!')
    
    # Local'e aç (çok daha hızlı)
    !unzip -o -q {VELODYNE_ZIP} -d {TEMP_DIR}
    
    # Drive'a taşı (bu kısım yavaş ama güvenli)
    print('Drive\'a kopyalanıyor...')
    !cp -r {TEMP_DIR}/training/velodyne/* {TRAIN_DIR}/velodyne/
    
    # Local temp'i temizle (disk dolmasın)
    !rm -rf {TEMP_DIR}/training/velodyne
    
    # Doğrula
    num_files = len(os.listdir(f'{TRAIN_DIR}/velodyne'))
    print(f'  Velodyne tamam: {num_files} dosya')
else:
    num_files = len(os.listdir(f'{TRAIN_DIR}/velodyne'))
    print(f'Velodyne zaten mevcut: {num_files} dosya, atlanıyor.')

In [ ]:
%%time
# --- 4. Images (OPSİYONEL — camera fusion yapmayacaksan atla) ---

image_check = f'{TRAIN_DIR}/image_2/000000.png'

if os.path.exists(IMAGE_ZIP) and not os.path.exists(image_check):
    print('Images açılıyor... (opsiyonel, ~10-20 dk)')
    !unzip -o -q {IMAGE_ZIP} -d {TEMP_DIR}
    !cp -r {TEMP_DIR}/training/image_2/* {TRAIN_DIR}/image_2/
    !rm -rf {TEMP_DIR}/training/image_2
    num_files = len(os.listdir(f'{TRAIN_DIR}/image_2'))
    print(f'  Images tamam: {num_files} dosya')
elif os.path.exists(image_check):
    print('Images zaten mevcut, atlanıyor.')
else:
    print('Image zip bulunamadı, atlanıyor (opsiyonel).')

# Temp klasörünü temizle
!rm -rf {TEMP_DIR}
print('Temp temizlendi.')

## Train/Val Split Oluşturma

In [ ]:
import numpy as np

IMAGESET_DIR = f'{KITTI_DIR}/ImageSets'
os.makedirs(IMAGESET_DIR, exist_ok=True)

# Mevcut velodyne dosyalarından ID'leri al
vel_dir = f'{TRAIN_DIR}/velodyne'
all_ids = sorted([f.replace('.bin', '') for f in os.listdir(vel_dir) if f.endswith('.bin')])
print(f'Toplam sample: {len(all_ids)}')

# Standard split: %50 train, %50 val (KITTI convention)
np.random.seed(0)
indices = np.random.permutation(len(all_ids))
split_point = len(all_ids) // 2

train_ids = sorted([all_ids[i] for i in indices[:split_point]])
val_ids = sorted([all_ids[i] for i in indices[split_point:]])

with open(f'{IMAGESET_DIR}/train.txt', 'w') as f:
    f.write('\n'.join(train_ids))

with open(f'{IMAGESET_DIR}/val.txt', 'w') as f:
    f.write('\n'.join(val_ids))

print(f'Train: {len(train_ids)} samples')
print(f'Val:   {len(val_ids)} samples')
print(f'Split dosyaları: {IMAGESET_DIR}')

## Doğrulama

In [ ]:
# Her şeyin yerinde olduğunu kontrol et
print('=== KITTI Dataset Doğrulama ===')
print()

checks = {
    'Velodyne': (f'{TRAIN_DIR}/velodyne', '.bin'),
    'Calib':    (f'{TRAIN_DIR}/calib', '.txt'),
    'Labels':   (f'{TRAIN_DIR}/label_2', '.txt'),
    'Images':   (f'{TRAIN_DIR}/image_2', '.png'),
}

all_ok = True
for name, (path, ext) in checks.items():
    if os.path.exists(path):
        count = len([f for f in os.listdir(path) if f.endswith(ext)])
        status = 'OK' if count > 0 else 'BOSSS'
        print(f'  {name:10s}: {count:6d} dosya  [{status}]')
        if count == 0 and name != 'Images':
            all_ok = False
    else:
        optional = ' (opsiyonel)' if name == 'Images' else ' *** EKSIK ***'
        print(f'  {name:10s}: BULUNAMADI{optional}')
        if name != 'Images':
            all_ok = False

# Train/val split
for split_name in ['train', 'val']:
    split_file = f'{KITTI_DIR}/ImageSets/{split_name}.txt'
    if os.path.exists(split_file):
        with open(split_file) as f:
            count = len(f.readlines())
        print(f'  {split_name:10s}: {count:6d} samples [OK]')
    else:
        print(f'  {split_name:10s}: BULUNAMADI')
        all_ok = False

# Disk kullanımı
print()
!du -sh {KITTI_DIR}

print()
if all_ok:
    print('KITTI dataset hazır! Artık 01_colab_setup.ipynb ile eğitime başlayabilirsin.')
else:
    print('EKSIK DOSYALAR VAR! Yukarıdaki adımları kontrol et.')

In [ ]:
# Bonus: Bir sample yükleyip görselleştirelim (her şey doğru mu?)
import numpy as np
import matplotlib.pyplot as plt

sample_id = train_ids[0]
points = np.fromfile(f'{TRAIN_DIR}/velodyne/{sample_id}.bin', dtype=np.float32).reshape(-1, 4)

print(f'Sample: {sample_id}')
print(f'Points: {points.shape[0]:,} nokta')
print(f'X range: [{points[:,0].min():.1f}, {points[:,0].max():.1f}] m')
print(f'Y range: [{points[:,1].min():.1f}, {points[:,1].max():.1f}] m')
print(f'Z range: [{points[:,2].min():.1f}, {points[:,2].max():.1f}] m')

# BEV plot
fig, ax = plt.subplots(figsize=(10, 8))
mask = (points[:,0] > 0) & (points[:,0] < 70) & (np.abs(points[:,1]) < 40)
ax.scatter(points[mask, 0], points[mask, 1], c=points[mask, 2], s=0.1, cmap='viridis', alpha=0.5)
ax.set_xlabel('X (m) — ileri')
ax.set_ylabel('Y (m) — sol/sağ')
ax.set_title(f'KITTI BEV — Sample {sample_id}')
ax.set_aspect('equal')
plt.colorbar(ax.collections[0], label='Z (m)')
plt.tight_layout()
plt.show()

# Label kontrol
with open(f'{TRAIN_DIR}/label_2/{sample_id}.txt') as f:
    labels = f.readlines()
print(f'\nLabels ({len(labels)} obje):')
for l in labels[:5]:
    print(f'  {l.strip()}')